### A. where behavior, ripple, parsed replay data lives

In [1]:
%reload_ext autoreload
%autoreload 2

In [3]:
import os
import pickle
import spyglass as nd
import pandas as pd
import statsmodels.api as sm
# ignore datajoint+jupyter async warnings
import warnings
warnings.simplefilter('ignore', category=DeprecationWarning)
warnings.simplefilter('ignore', category=ResourceWarning)

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import logging
import multiprocessing

FORMAT = '%(asctime)s %(message)s'

logging.basicConfig(level='INFO', format=FORMAT, datefmt='%d-%b-%y %H:%M:%S')

from spyglass.common import (Session, IntervalList,IntervalPositionInfo,
                             LabMember, LabTeam, Raw, Session, Nwbfile,
                            Electrode,LFPBand,interval_list_intersect)
from spyglass.common.common_interval import _intersection

#from spyglass.common.nwb_helper_fn import get_nwb_copy_filename
from spyglass.common.common_position import IntervalPositionInfo, IntervalPositionInfoSelection


# Here are the analysis tables specific to Shijie Gu
from spyglass.shijiegu.Analysis_SGU import (TrialChoice,
                                            TrialChoiceReplay,
                                            RippleTimes,
                                            Decode,
                                            TrialChoiceReplayTransition,
                                            get_linearization_map,
                                            find_ripple_times,classify_ripples,classify_ripple_content)
from spyglass.shijiegu.PastFuture_Replay import (replay_in_categories,find_distinct_subset,proportion,
                                                 unravel_replay,count_replay_by_category,category_day)

In [4]:
pd.set_option('display.max_rows', None)

### 1. Animal's behavior + classified replay contents + replayed arm transitions
These data are in the Table ```TrialChoiceReplayTransition```in the database. In the subfield ```choice_reward_replay_transition```

Parsed replayed transitions at the home well for each trial is in the column ```replayed_transitions```

IMPORTANT: Note that only legal trials are included. Feel free to use this version of behavior dataset.

In [5]:
'''
Just take a look at the table. 
With the two entries "nwb_file_name" and "epoch",
    you can retrieve other information in the table.
    See more in the cells below.
'''
TrialChoiceReplayTransition()

nwb_file_name name of the NWB file,epoch the session epoch for this task and apparatus(1 based),"epoch_name session name, get from IntervalList","transitions ndarray, for all replayed arm transitions","choice_reward_replay_transition pandas dataframe, choice, reward, replayed arm transitions, ripple time, replays"
molly20220415_.nwb,2,02_SeqSession1,=BLOB=,=BLOB=
molly20220415_.nwb,4,04_Seq2Session1,=BLOB=,=BLOB=
molly20220415_.nwb,6,06_Seq2Session2,=BLOB=,=BLOB=
molly20220415_.nwb,8,08_Seq2Session3,=BLOB=,=BLOB=
molly20220416_.nwb,2,02_Seq2Session1,=BLOB=,=BLOB=
molly20220416_.nwb,4,04_Seq2Session2,=BLOB=,=BLOB=
molly20220416_.nwb,6,06_Seq2Session3,=BLOB=,=BLOB=
molly20220416_.nwb,8,08_Seq2Session4,=BLOB=,=BLOB=
molly20220416_.nwb,10,10_Seq2Session5,=BLOB=,=BLOB=
molly20220417_.nwb,2,02_Seq2Session1,=BLOB=,=BLOB=


In [6]:
'''choose a day'''
nwb_copy_file_name='molly20220420_.nwb'
all_epochs=list((TrialChoiceReplayTransition() & 
                 {'nwb_file_name':nwb_copy_file_name}).fetch('epoch'))
print('epochs all this day:',all_epochs)

epochs all this day: [2, 4, 6, 8, 10, 12]


In [7]:
'''choose an epoch'''
epoch_num = 4

In [8]:
'''get the behavior + ripple/replay + parsed replay data on this day and this epoch'''
T=pd.DataFrame((TrialChoiceReplayTransition() & {'nwb_file_name':nwb_copy_file_name,
                                 'epoch':epoch_num}).fetch1('choice_reward_replay_transition'))
T

,timestamp_H,Home,timestamp_O,OuterWellIndex,replayed_transitions,rewardNum,ripple_H,ripple_O,replay_H,ripple_peak_H,replay_O,ripple_peak_O
1,1.650477e+09,1.0,1.650478e+09,3.0,[],2.0,"[[[1650477506.9091513, 1650477506.9531515]], [...","[[[1650477522.7511444, 1650477522.7971444]], [...","[[nan], [nan, 0.0, nan], [0]]",[],"[[3], [3], [3], [3], [3], [3], [nan]]",[]
2,1.650478e+09,1.0,1.650478e+09,4.0,[],2.0,"[[[1650477535.4171388, 1650477535.5491385]], [...","[[[1650477543.0431354, 1650477543.1231353]], [...","[[0], [0], [0], [0], [0]]",[],"[[4], [4], [4], [5.0, nan], [4], [4], [4.0, na...",[]
5,1.650478e+09,1.0,1.650478e+09,3.0,[],1.0,"[[[1650477586.973116, 1650477587.0931158]], [[...","[[[1650477593.7071128, 1650477593.7911127]]]","[[0], [0]]",[],[[3]],[]
6,1.650478e+09,1.0,1.650478e+09,4.0,[],1.0,"[[[1650477598.3211107, 1650477598.4211106]], [...",[],"[[0], [0], [0]]",[],[],[]
7,1.650478e+09,1.0,1.650478e+09,2.0,[],1.0,"[[[1650477610.7611053, 1650477610.8471053]], [...","[[[1650477620.5791008, 1650477620.9031007]], [...","[[0], [0], [0]]",[],"[[2], [2], [2], [2], [2], [2], [2], [2]]",[]
8,1.650478e+09,1.0,1.650478e+09,1.0,[],2.0,"[[[1650477636.4330938, 1650477636.6090937]], [...","[[[1650477644.5390902, 1650477644.6130903]], [...","[[0], [0], [0], [0], [0], [0], [0], [0]]",[],"[[1], [1], [5, 1], [1], [1.0, nan], [1], [1], ...",[]
9,1.650478e+09,1.0,1.650478e+09,4.0,[],1.0,"[[[1650477660.0790832, 1650477660.1370833]], [...","[[[1650477668.1990798, 1650477668.2790797]]]","[[0], [0], [0]]",[],[[4]],[]
10,1.650478e+09,1.0,1.650478e+09,3.0,[],2.0,"[[[1650477674.495077, 1650477674.533077]], [[1...","[[[1650477682.0570736, 1650477682.1010735]], [...","[[0], [0], [0], [0], [0]]",[],"[[3], [3], [3], [3], [3], [3], [3], [3], [3], ...",[]
11,1.650478e+09,1.0,1.650478e+09,4.0,[],2.0,"[[[1650477700.7430654, 1650477700.8490653]], [...","[[[1650477715.8290586, 1650477715.8790586], [1...","[[0], [0], [0], [nan, 5.0], [5], [0], [0], [0]]",[],"[[4.0, nan, 4.0, nan], [3], [4]]",[]
14,1.650478e+09,1.0,1.650478e+09,3.0,[],1.0,"[[[1650477760.1910388, 1650477760.2830389]], [...",[],"[[0], [0], [0], [0], [0], [0], [4], [0], [0], ...",[],[],[]


### 2. all replayed arm transitions and all replayed singleton arm count
These data are in the Table ```TrialChoiceReplayTransition```in the database. Subfield ```transitions```

Entry [i,j] means the number of a replay event of arm i followed by arm j within 200 ms, when i not equal to j.


The diagonal entries [i,i] are the number of singleton replays of arm i. The diagonal entries are over-counted due to the lick artifact in the neural data splitting a smooth replay into 2. Take the diagonal entries as a grain of salt for Molly's data.

In [9]:
transitions_all=(TrialChoiceReplayTransition() &
                 {'nwb_file_name':nwb_copy_file_name,
                  'epoch':epoch_num}).fetch1('transitions')
transitions_all

array([[17.,  3.,  0.,  0.],
       [ 5., 21.,  0.,  3.],
       [ 1.,  2., 13.,  0.],
       [ 0.,  1.,  0., 18.]])

### 3. Animal's behavior + annotated future, past, past reward arms
These data is in the table called ```TrialChoice()``` in the database.


IMPORTANT: Note that ALL trials are included, as opposed to legal trials only.


In [10]:
TrialChoice()

nwb_file_name name of the NWB file,epoch the session epoch for this task and apparatus(1 based),"epoch_name session name, get from IntervalList","choice_reward pandas dataframe, choice"
molly20220415_.nwb,2,02_SeqSession1,=BLOB=
molly20220415_.nwb,4,04_Seq2Session1,=BLOB=
molly20220415_.nwb,6,06_Seq2Session2,=BLOB=
molly20220415_.nwb,8,08_Seq2Session3,=BLOB=
molly20220416_.nwb,2,02_Seq2Session1,=BLOB=
molly20220416_.nwb,4,04_Seq2Session2,=BLOB=
molly20220416_.nwb,6,06_Seq2Session3,=BLOB=
molly20220416_.nwb,8,08_Seq2Session4,=BLOB=
molly20220416_.nwb,10,10_Seq2Session5,=BLOB=
molly20220417_.nwb,2,02_Seq2Session1,=BLOB=


#### Notes:
At home well: 
- Future arm is in the column ```future_H```, which corresponds to the arm the rat will choose on this trial.
    
At arm well: 

- Future arm is in the column ```future_O```, which corresponds to the arm the rat will choose on the next trial.
    
For both home and arm well:
    
   - Past arm is in the column ```past```, which corresponds to the arm the rat was at on the last trial.
   
   - Past rewarded arm is in the column ```past_reward```, which corresponds to the arm the rat was at on the last trial, and in addition, he got a reward there.

In [11]:
T=(TrialChoice() &{'nwb_file_name':nwb_copy_file_name,
                  'epoch':epoch_num}).fetch1('choice_reward')

In [12]:
T=pd.DataFrame(T)
T

,timestamp_H,Home,timestamp_O,OuterWellIndex,rewardNum,current,future_H,future_O,past,past_reward
1,1.650477e+09,1.0,1.650478e+09,3.0,2.0,3.0,3.0,4.0,NaN,NaN
2,1.650478e+09,1.0,1.650478e+09,4.0,2.0,4.0,4.0,2.0,3.0,3.0
3,1.650478e+09,1.0,1.650478e+09,2.0,2.0,2.0,2.0,1.0,4.0,4.0
4,NaN,NaN,1.650478e+09,1.0,0.0,1.0,1.0,3.0,2.0,2.0
5,1.650478e+09,1.0,1.650478e+09,3.0,1.0,3.0,3.0,4.0,1.0,2.0
6,1.650478e+09,1.0,1.650478e+09,4.0,1.0,4.0,4.0,2.0,3.0,2.0
7,1.650478e+09,1.0,1.650478e+09,2.0,1.0,2.0,2.0,1.0,4.0,2.0
8,1.650478e+09,1.0,1.650478e+09,1.0,2.0,1.0,1.0,4.0,2.0,2.0
9,1.650478e+09,1.0,1.650478e+09,4.0,1.0,4.0,4.0,3.0,1.0,1.0
10,1.650478e+09,1.0,1.650478e+09,3.0,2.0,3.0,3.0,4.0,4.0,1.0


In [13]:
# to find future choice on trial 5 for example.
T.loc[5,'future_H'] 

3.0